In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_data.csv")

In [2]:
df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
df['TransactionAmt_decimal'] = df['TransactionAmt'] % 1
df['TransactionAmt_isround'] = (df['TransactionAmt'] % 1 == 0).astype(int)

In [3]:
df['hour'] = df['TransactionDT'] % 86400 // 3600
df['day']  = df['TransactionDT'] // 86400 % 7
df['is_night'] = df['hour'].apply(lambda x: 1 if x < 6 or x > 22 else 0)

In [4]:
# How frequently does each card appear?
# High frequency = more data = better fraud detection
df['card1_freq'] = df['card1'].map(df['card1'].value_counts())
df['card2_freq'] = df['card2'].map(df['card2'].value_counts())

# Card + amount combination
df['card1_amt_mean'] = df.groupby('card1')['TransactionAmt'].transform('mean')
df['card1_amt_std']  = df.groupby('card1')['TransactionAmt'].transform('std')

# How different is this transaction from card's normal amount?
df['amt_vs_card1_mean'] = df['TransactionAmt'] / (df['card1_amt_mean'] + 1)

In [5]:
# Fill missing
df['P_emaildomain'] = df['P_emaildomain'].fillna('unknown')

# Email provider category
def email_category(email):
    if email in ['gmail.com', 'yahoo.com', 'hotmail.com']:
        return 'common'
    elif email == 'unknown':
        return 'unknown'
    else:
        return 'other'

df['email_category'] = df['P_emaildomain'].apply(email_category)

In [6]:
# Frequency of address
df['addr1_freq'] = df['addr1'].map(df['addr1'].value_counts())

# Address + card combination — mismatch can indicate fraud
df['addr1_card1_freq'] = df.groupby(['addr1', 'card1'])['TransactionID'].transform('count')

In [7]:
# D columns are days between events
# Fill with -1 to indicate no previous transaction
d_cols = [col for col in df.columns if col.startswith('D')]

for col in d_cols:
    df[col] = df[col].fillna(-1)

# D1 is very important — days since last transaction on card
df['D1_normalized'] = df['D1'] / df['D1'].max()

In [8]:
# C columns are counts — already useful as is
# Just fill missing values
c_cols = [col for col in df.columns if col.startswith('C')]

for col in c_cols:
    df[col] = df[col].fillna(0)

# Total count across all C columns
df['C_sum'] = df[c_cols].sum(axis=1)

In [9]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()

for col in cat_cols:
    df[col] = df[col].fillna('unknown')
    df[col] = le.fit_transform(df[col].astype(str))

In [10]:
# Numerical columns fill with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

In [15]:
df.columns

Index(['isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1',
       'card2', 'card3', 'card4', 'card5', 'card6',
       ...
       'card1_freq', 'card2_freq', 'card1_amt_mean', 'card1_amt_std',
       'amt_vs_card1_mean', 'email_category', 'addr1_freq', 'addr1_card1_freq',
       'D1_normalized', 'C_sum'],
      dtype='object', length=235)

In [16]:


df.to_csv('../data/processed/featured_data.csv', index=False)

print(f"Final shape: {df.shape}")
print(f"Fraud ratio: {df['isFraud'].mean():.4f}")

Final shape: (590540, 235)
Fraud ratio: 0.0350
